In [ ]:
import os

import torch

import pandas as pd 
import sqlite3

from pathlib import Path

ROOT = Path.cwd().parents[1]

EMBED_PATH = ROOT / "data/embeddings/base_embeds.pt"
EMBED_NAME = EMBED_PATH.stem

IMAGE_DIR = ROOT / "images/ellipsoid" / EMBED_NAME
CACHE_DIR = ROOT / "data/cache" / EMBED_NAME
RESULTS_DIR = ROOT / "data/results" / EMBED_NAME

In [3]:
os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

In [4]:
embeds = torch.load(EMBED_PATH, weights_only=False)
cls_tokens = embeds["cls_tokens"]
patches = embeds["patches"]

In [5]:
conn = sqlite3.connect(ROOT / "data/sql/metadata.db")

meta = pd.read_sql_query("SELECT * FROM meta", conn)
categories = pd.read_sql_query("SELECT DISTINCT category FROM meta", conn)["category"].to_list()
types_per_cat = pd.read_sql_query("""
    SELECT category, COUNT(DISTINCT type) AS num_types
    FROM meta
    GROUP BY category
""", conn)
max_types = types_per_cat["num_types"].max()

conn.close()

In [ ]:
train_mask = meta["split"] == "train"

train_cls = cls_tokens[train_mask]
test_cls = cls_tokens[~train_mask]

train_meta = meta[train_mask]
test_meta = meta[~train_mask]

for cat in categories:
    train_cat_mask = train_meta["category"]  == cat
    test_cat_mask = test_meta["category"] == cat

    train_emb = train_cls[train_cat_mask]
    test_emb = test_cls[test_cat_mask]

  


[0 1 0 1 1 2 0 0 1 2 1 0 0 0 2 2 1 1 1 0 1 0 0 0 0 2 1 1 0 1 0 1 2 1 1 1 2
 2 0 0 0 1 2 1 0 0 1 1 0 0 1 2 0 0 1 0 0 2 0 2 2 0 1 0 1 2 2 1 0 2 2 0 0 1
 1 1 0 1 0 1 0 1 2 1 0 2 1 0 0 1 0 0 2 0 2 1 2 2 1 0 2 1 2 2 0 0 2 2 1 2 2
 1 1 1 2 1 0 0 1 1 2 2 1 0 0 2 1 0 1 2 2 1 2 2 1 1 0 1 1 1 2 2 0 0 0 1 1 1
 1 2 1 0 2 2 1 0 1 0 1 1 1 0 0 0 1 1 2 1 2 1 0 0 2 2 1 1 1 2 2 2 2 0 1 0 1
 2 0 2 0 2 2 2 1 0 2 1 2 0 1 1 1 1 0 0 2 2 2 0 2]


ValueError: Fitting the mixture model failed because some components have ill-defined empirical covariance (for instance caused by singleton or collapsed samples). Try to decrease the number of components, increase reg_covar, or scale the input data. The numerical accuracy can also be improved by passing float64 data instead of float32.